# Agent with multiple tools


### SETUP
Install Langchain dependencies

In [ ]:
%pip install -U langchain langchain-core langchain-community langchain-ollama youtube_search tavily-python langsmith --quiet

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from tavily import TavilyClient
from IPython.display import Markdown

C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Read Tavily credentials and setup client 

In [2]:
import os
from dotenv import load_dotenv
load_dotenv() #.env includes Langsmith and Tavily API keys
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

### Tools

In [3]:
@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    response = tavily_client.search(query=f"current weather in {city}", search_depth="basic")
    return response['results']

@tool
def get_clothing_advice(weather: str) -> str:
    """Get clothing advice depending on the weather. Possible weather conditions are: sunny, rainy, snowy, cloudy."""
    advice = {
        "sunny": "Wear light clothing, sunglasses, and sunscreen.",
        "rainy": "Wear a waterproof jacket and carry an umbrella.",
        "snowy": "Wear a heavy coat, gloves, and a hat.",
        "cloudy": "Wear comfortable clothing and consider a light jacket.",
    }
    return advice.get(weather, "No specific advice available for this weather condition.")

@tool
def web_search(query: str) -> str:
    """Perform a web search for a given query."""
    response = tavily_client.search(query=query, max_results=5, search_depth="basic")
    return response['results']

Model and agent

In [4]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

SYSTEM_PROMPT = """You are a helpful assistant.
You have access to these tools:
- get_weather: use this to get the weather for a specific location
- get_clothing_advice: use this to get clothing advice based on weather conditions
- web_search: use this to perform a web search for a given query"""

checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[get_weather, get_clothing_advice, web_search],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)

### TESTING
Let's try it out. First question

In [5]:
query = "how many cities in the world are called Brisbane and what country are they in? Include only cities with more than 3,000 inhabitants."
response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"configurable": {"thread_id": "001"}}
)

In [6]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='how many cities in the world are called Brisbane and what country are they in? Include only cities with more than 3,000 inhabitants.', additional_kwargs={}, response_metadata={}, id='0cc45b2e-39f7-4d8e-8141-b1f368771160'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-11T00:53:35.3059033Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4015113300, 'load_duration': 285022800, 'prompt_eval_count': 260, 'prompt_eval_duration': 279088600, 'eval_count': 291, 'eval_duration': 3276312900, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e1486-6016-7383-9922-38d0e913d8cb-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'cities named Brisbane with population over 3000'}, 'id': '48f0552f-b9fc-46f9-8f95-bb7ab27aee3f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 260, 'output_tokens': 291, 'total_

In [7]:
Markdown(response["messages"][-1].content)

Based on the web search results, there are at least **two** major locations named Brisbane in the world, found in **Australia** and the **United States**.

The search results indicate:
*   **Australia:** Brisbane, Queensland (This location is shown with a very large population, exceeding 3,000 inhabitants).
*   **United States:** There are multiple locations mentioned in the US (such as California, Illinois, and North Dakota), but the search results do not provide a definitive count or population data for all of them to confirm which ones exceed 3,000 inhabitants.

To get a definitive, up-to-date list that strictly adheres to the population requirement, you may need to perform a more specific search or consult a dedicated geographical database.

![Langchain workflow](images/langsmith-multitool-run1.png)

Second question

In [8]:
query = "If I want to visit today these 2 cities named Brisbane you found, what should I wear?"
response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"configurable": {"thread_id": "001"}}
)

In [9]:
Markdown(response["messages"][-1].content)

Based on the weather information I found for **Brisbane, Australia**, the forecast suggests a high chance of rain ("Patchy rain nearby").

Therefore, I recommend you **wear a waterproof jacket and carry an umbrella** for your visit today.

***

**Please note:** The weather information I retrieved was only for **Brisbane, Australia**. Since I do not have the current weather for the other potential Brisbane location in the United States, please check the local forecast for that city before you go!

In [10]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='how many cities in the world are called Brisbane and what country are they in? Include only cities with more than 3,000 inhabitants.', additional_kwargs={}, response_metadata={}, id='0cc45b2e-39f7-4d8e-8141-b1f368771160'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-11T00:53:35.3059033Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4015113300, 'load_duration': 285022800, 'prompt_eval_count': 260, 'prompt_eval_duration': 279088600, 'eval_count': 291, 'eval_duration': 3276312900, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e1486-6016-7383-9922-38d0e913d8cb-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'cities named Brisbane with population over 3000'}, 'id': '48f0552f-b9fc-46f9-8f95-bb7ab27aee3f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 260, 'output_tokens': 291, 'total_

In [11]:
for m in response["messages"]:
    print(m.pretty_print())

================================ Human Message =================================

how many cities in the world are called Brisbane and what country are they in? Include only cities with more than 3,000 inhabitants.
None
================================== Ai Message ==================================
Tool Calls:
  web_search (48f0552f-b9fc-46f9-8f95-bb7ab27aee3f)
 Call ID: 48f0552f-b9fc-46f9-8f95-bb7ab27aee3f
  Args:
    query: cities named Brisbane with population over 3000
None
================================= Tool Message =================================
Name: web_search

[{"url": "https://www.geodatos.net/en/homonymous-cities/brisbane", "title": "Cities named Brisbane around the world", "content": "Scripts are disabled by your browser. Some features of this website will not work. # Cities named Brisbane. There are at least **2 cities named Brisbane** in the entire world. The following map and table contain general information about these cities. | 1 | United States | California | 

![Langchain workflow](images/langsmith-multitool-run2.png)